In [ ]:
import qutip as qt
from qutip import tensor, basis, qeye, Qobj
import numpy as np
from quantum_logical.trotter_diff import Trotterization
from tqdm import tqdm
from quantum_logical.state import state as st
from quantum_logical.state import combined_state
from quantum_logical.cnot_gate_creation import cnot

# there is circular importation that needs fixed 

In [ ]:
def gate(dim, N):
    from quantum_logical.gate_extender import Gate_extender, Convert_levels
    from quantum_logical.cnot_gate_creation import cnot
    # creating the gates
    dim = dim
    N = N
    # creating the set of cnots (this will operate between the g and e levels)

    cnot1 = cnot(N=N, target=3, control=0, high=1, low=0)
    cnot2 = cnot(N=N, target=3, control=1, high=1, low=0)
    
    cnot3 = cnot(N=N, target=4, control=1, high=1, low=0)
    cnot4 = cnot(N=N, target=4, control=2, high=1, low=0)

    # the x_gate needs to be made in a qutrit gate and will involve conversion 
    x_gate = qt.Qobj([[0, 1],[1, 0]])

    hada = qt.Qobj([[1/np.sqrt(2), 0, 1/np.sqrt(2)], [0, 1, 0], [1/np.sqrt(2), 0, -1/np.sqrt(2)]])

    gate_extention = Gate_extender(num_qubits=1)
    x_gate_ = gate_extention.qubit_to_qudit(gate=x_gate, from_dim=2, to_dim=dim)


    # conversion of some of the gates into the qutrit space 
    new_dim = dim
    converter = Convert_levels(num_qubits=1)
    x_gate = converter.level_conversion(levels=[0,2], dim=dim, gate=x_gate_, qubits=None)


    x_layer = tensor(tensor([x_gate] * 3), tensor([qeye(new_dim)] * 2))
    hada_layer = tensor(tensor([hada] * 3), tensor([qeye(new_dim)] * 2))

    # building the correction z_gate 
    correction_x = qt.Qobj([[0, 1],[1, 0]])
    gate_extention = Gate_extender(num_qubits=1)
    correction_x = gate_extention.qubit_to_qudit(gate=correction_x, from_dim=2, to_dim=3)
    converter = Convert_levels(num_qubits=1)
    correction_x = converter.level_conversion(levels=[1,2], dim=new_dim, gate=correction_x, qubits=None)

    correction_z = (hada * correction_x * hada.dag())


    cnots = [cnot1, cnot2, cnot3, cnot4]
    return cnots, correction_z, hada_layer, x_layer, correction_x, x_gate, hada, x_gate_

In [ ]:
# def repetition(values, rho, total_time):
    
#     cnots, T, dim, N, x_gate, x_layer, correction_z, _, hada, _ = values # collecting experimental values 

#     # initialize the state with ancillas 
#     rho = tensor(qt.ptrace(rho, [0,1,2]), 
#                  tensor(basis(dim, 0), basis(dim, 0)) * tensor(basis(dim, 0),
#                  basis(dim, 0)).dag())

#     hada_layer_mod = tensor([hada] * 3)

#     # gate instructions
#     gates = [[hada_layer_mod], [x_layer], [cnots[0]], [tensor(x_gate, tensor([qeye(dim)] * 4))],
#              [tensor(hada, tensor([qeye(dim)] * 4))] ,[cnots[1]],
#              [cnots[2]], [tensor(qeye(dim), x_gate, tensor([qeye(dim)] * 3))], 
#              [tensor(qeye(dim), hada, tensor([qeye(dim)] * 3))],
#              [cnots[3]], [tensor(tensor([qeye(dim)]*2), x_gate, tensor([qeye(dim)] * 2))], 
#              [tensor(tensor([qeye(dim)]*2), hada, tensor([qeye(dim)] * 2))]]
#     cnot_time = .5
#     gate_times = [.03, .03, cnot_time, .03, .03, cnot_time, cnot_time, .03, .03, cnot_time, .03, .03]

#     for i in range(len(gates)):
#         if i == 0: # ensures that the hadamard can be fractionally broken and used 
#             trotter_dt = gate_times[i] / 20
#             trotter = Trotterization(trotter_dt=trotter_dt, T1=T[0], T2=T[1], dim=dim,
#                                      num_qubits=3, qudit="qutrit")
#             rho_enc = trotter.apply(rho=qt.ptrace(rho, [0,1,2]), duration=gate_times[i],
#                                     unitary=gates[i], errors=True)
#             rho = tensor(rho_enc[-1], 
#                          tensor(basis(dim, 0), basis(dim, 0)) * tensor(basis(dim, 0), basis(dim, 0)).dag()) # Ensures dimensionality for future gates
#         else: 
#             trotter_dt = gate_times[i] / 20
#             trotter = Trotterization(trotter_dt=trotter_dt, T1=T[0], T2=T[1], dim=dim,
#                                      num_qubits=N, qudit="qutrit")
#             rho_enc = trotter.apply(rho=rho, duration=gate_times[i], unitary=gates[i],
#                                     errors=True)
#             rho = rho_enc[-1]
#         total_time += gate_times[i]


#     # projection operators  
#     proj = [qt.tensor(qt.qeye(dim), qt.qeye(dim), qt.qeye(dim), 
#             (qt.tensor(qt.basis(dim, i), qt.basis(dim, j)) * (qt.tensor(qt.basis(dim, i), qt.basis(dim, j))).dag())) 
#             for i in range(3) for j in range(3)]


#     # correction_operators
#     r00 = r02 = r20 = r22 = r12 = r21 = qt.tensor([qt.qeye(dim)] * 5)
#     r01 = qt.tensor(qt.qeye(dim), qt.qeye(dim), correction_z, qt.tensor([qt.qeye(dim)] * 2))
#     r10 = qt.tensor(correction_z, qt.tensor([qt.qeye(dim)] * 4))
#     r11 = qt.tensor(qt.qeye(dim), correction_z, qt.qeye(dim), qt.tensor([qt.qeye(dim)] * 2))
#     recovery_ops = [[r00], [r01], [r02], [r10], [r11], [r12], [r20], [r21], [r22]]

#     # measurement 
#     measurement_duration = 2
#     trotter_dt = measurement_duration / 20
#     trotter = Trotterization(trotter_dt=trotter_dt, T1=T[0], T2=T[1], dim=dim, num_qubits=N, qudit="qutrit")
#     measurement_delay_evo = trotter.apply(rho=rho, duration=measurement_duration, 
#                                           unitary=[tensor([qt.qeye(dim)] * N)], errors=True)

#     # building branches based on measurement 
#     proj_results_after_measurement = [(measurement_delay_evo[-1] * proj).tr() for proj in proj]
#     proj_states_after_measurment = [(proj * measurement_delay_evo[-1] * proj.dag()) for proj in proj]
#     total_time += measurement_duration

#     # corrects the branches after measurement 
#     corrected_states = []
#     recovery_duration = .03
#     for i in range(len(recovery_ops)):
#         if proj_results_after_measurement[i] != 0: # ensures no division by zero in trotterization
#             trotter_dt = recovery_duration / 20
#             trotter = Trotterization(trotter_dt=trotter_dt, T1=T[0], T2=T[1], dim=dim, num_qubits=N, qudit="qutrit")
#             corrected_state = trotter.apply(proj_states_after_measurment[i], duration=recovery_duration, 
#                                             unitary=recovery_ops[i], errors=True)
#             corrected_states.append(corrected_state[-1])
#         else:
#             corrected_states.append(recovery_ops[i][0] * proj_states_after_measurment[i] * recovery_ops[i][0].dag())
#     total_time += recovery_duration

#     # recombining branches post correction
#     repetition_corrected_state = (sum([proj_results_after_measurement[j] * corrected_states[j] 
#                                        for j in range(len(proj_results_after_measurement))]) / 
#                                        sum([proj_results_after_measurement[j] * corrected_states[j] 
#                                        for j in range(len(proj_results_after_measurement))]).tr())

#     return repetition_corrected_state, total_time

In [ ]:
def repetition(values, rho, total_time):

    cnots, T, dim, N, x_gate, _, correction_z, _, hada, _ = values # collecting experimental values 

    # initialize the state with ancillas 
    rho = tensor(qt.ptrace(rho, [0,1,2]), 
                 tensor(basis(dim, 0), basis(dim, 0)) * tensor(basis(dim, 0), basis(dim, 0)).dag())

    # if cycle_count == 0:
    gates = [[tensor(hada, hada, tensor([qeye(dim)] * 1))],
             [tensor(x_gate, x_gate, tensor([qeye(dim)] * 3))], 
             [cnots[0]], [tensor(x_gate, tensor([qeye(dim)] * 4))], [tensor(hada, tensor([qeye(dim)] * 4))],
             [cnots[1]], [cnots[2]], [tensor(qeye(dim), x_gate, hada, tensor([qeye(dim)] * 2))],
             [tensor(qeye(dim), hada, tensor([qeye(dim)] * 3)), tensor(qeye(dim), qeye(dim), x_gate, tensor([qeye(dim)] * 2))],
             [cnots[3]], [tensor(tensor([qeye(dim)]*2), x_gate, tensor([qeye(dim)] * 2))],
             [tensor(tensor([qeye(dim)]*2), hada, tensor([qeye(dim)] * 2))]]
    
    cnot_time = .5
    gate_times = [.03, .03, cnot_time, .03, .03, cnot_time, cnot_time, .03, .03, cnot_time, .03, .03]

    for i in range(len(gates)):
        if i == 0: # ensures that the hadamard can be fractionally broken and used 
            trotter_dt = gate_times[i] / 20
            trotter = Trotterization(trotter_dt=trotter_dt, T1=T[0], T2=T[1], dim=dim, num_qubits=3, qudit="qutrit")
            rho_enc = trotter.apply(rho=qt.ptrace(rho, [0,1,2]), duration=gate_times[i], unitary=gates[i], errors=True)
            rho = tensor(rho_enc[-1], 
                         tensor(basis(dim, 0), basis(dim, 0)) * tensor(basis(dim, 0), basis(dim, 0)).dag()) # Ensures dimensionality for future gates
        else: 
            trotter_dt = gate_times[i] / 20
            trotter = Trotterization(trotter_dt=trotter_dt, T1=T[0], T2=T[1], dim=dim, num_qubits=N, qudit="qutrit")
            rho_enc = trotter.apply(rho=rho, duration=gate_times[i], unitary=gates[i], errors=True)
            rho = rho_enc[-1]
        total_time += gate_times[i]

    # projection operators  
    proj = [qt.tensor(qt.qeye(dim), qt.qeye(dim), qt.qeye(dim), 
            (qt.tensor(qt.basis(dim, i), qt.basis(dim, j)) * (qt.tensor(qt.basis(dim, i), qt.basis(dim, j))).dag())) 
            for i in range(3) for j in range(3)]


    # correction_operators
    r00 = r02 = r20 = r22 = r12 = r21 = qt.tensor([qt.qeye(dim)] * 5)
    r01 = qt.tensor(qt.qeye(dim), qt.qeye(dim), correction_z, qt.tensor([qt.qeye(dim)] * 2))
    r10 = qt.tensor(correction_z, qt.tensor([qt.qeye(dim)] * 4))
    r11 = qt.tensor(qt.qeye(dim), correction_z, qt.qeye(dim), qt.tensor([qt.qeye(dim)] * 2))
    recovery_ops = [[r00], [r01], [r02], [r10], [r11], [r12], [r20], [r21], [r22]]
    recovery_times = [[0], [.03], [0], [.03], [.03], [0], [0], [0], [0]]

    # measurement 
    measurement_duration = 2
    trotter_dt = measurement_duration / 20
    trotter = Trotterization(trotter_dt=trotter_dt, T1=T[0], T2=T[1], dim=dim, num_qubits=N, qudit="qutrit")
    measurement_delay_evo = trotter.apply(rho=rho, duration=measurement_duration, unitary=[tensor([qt.qeye(dim)] * N)], errors=True)

    # building branches based on measurement 
    proj_results_after_measurement = [(measurement_delay_evo[-1] * proj).tr() for proj in proj]
    proj_states_after_measurment = [(proj * measurement_delay_evo[-1] * proj.dag()) for proj in proj]
    total_time += measurement_duration

    # corrects the branches after measurement 
    corrected_states = []
    for i in range(len(recovery_ops)):
        if proj_results_after_measurement[i] != 0 and sum(recovery_times[i]) != 0.0: # ensures no division by zero in trotterization
            trotter_dt = recovery_times[i][0] / 20
            trotter = Trotterization(trotter_dt=trotter_dt, T1=T[0], T2=T[1], dim=dim, num_qubits=N, qudit="qutrit")
            corrected_state = trotter.apply(proj_states_after_measurment[i], duration=recovery_times[i][0], unitary=recovery_ops[i], errors=True)
            corrected_states.append(corrected_state[-1])
        else:
            corrected_states.append(recovery_ops[i][0] * proj_states_after_measurment[i] * recovery_ops[i][0].dag())
    
    for i in range(len(proj_results_after_measurement)):
        total_time += proj_results_after_measurement[i] * sum(recovery_times[i])

    # recombining branches post correction
    repetition_corrected_state = (sum([proj_results_after_measurement[j] * corrected_states[j] 
                                       for j in range(len(proj_results_after_measurement))]) / 
                                       sum([proj_results_after_measurement[j] * corrected_states[j] 
                                       for j in range(len(proj_results_after_measurement))]).tr())

    return repetition_corrected_state, total_time

In [ ]:
# def erasure(values, rho, total_time, state_choice=None):
    
    # _, T, dim, N, _, _, _, _, hada, x_gate = values  # experimental values

    # N = N + 1 

    # hada = qt.Qobj([[1/np.sqrt(2), 0, 1/np.sqrt(2)], [0, 1, 0], [1/np.sqrt(2), 0, -1/np.sqrt(2)]])

    # initial_state = qt.ptrace(rho, [0,1,2])
    # ancilla_states = tensor(basis(dim, 0), basis(dim, 0), basis(dim, 0)) * tensor(basis(dim, 0), basis(dim, 0), basis(dim, 0)).dag()
    # full_initial_state = tensor(initial_state, ancilla_states)

    # # detection gate creation
    # cnot1 = cnot(target=3, control=0, high=1, low=0, N=6)
    # cnot2 = cnot(target=4, control=1, high=1, low=0, N=6)
    # cnot3 = cnot(target=5, control=2, high=1, low=0, N=6)
    # # cnots = [cnot(target=t, control=c, high=1, low=0, N=6) for c,t in [[0,3], [1,4], [2,5]]]

    # gates = [cnot1, cnot2, cnot3]
    # cnot_time = .5
    # gate_times = [cnot_time, cnot_time, cnot_time]

    # # running the circuit
    # rho = full_initial_state

    # for i in range(len(gates)):
    #     trotter_dt = gate_times[i] / 20
    #     trotter = Trotterization(trotter_dt=trotter_dt, T1=T[0], T2=T[1], dim=dim, num_qubits=N, qudit="qutrit")
    #     rho_enc = trotter.apply(rho=rho, duration=gate_times[i], unitary=[gates[i]], errors=True)
    #     rho = rho_enc[-1]
    #     total_time += gate_times[i]
    
    # # measurement
    # # measurement operators 
    # proj = [tensor(qeye(dim), qeye(dim), qeye(dim), tensor(basis(dim, i), basis(dim, j), basis(dim, k))  
    #                * tensor(basis(dim, i), basis(dim, j), basis(dim, k)).dag()) 
    #          for i in [0,1] for j in [0,1] for k in [0,1]]
    
    # measurement_duration = 2
    # trotter_dt = measurement_duration / 20
    # trotter = Trotterization(trotter_dt=trotter_dt, T1=T[0], T2=T[1], dim=dim, num_qubits=N, qudit="qutrit")

    # measurement_evo = trotter.apply(rho=rho, duration=measurement_duration, unitary=[tensor([qeye(dim)] * N)], errors=True)
    # total_time += measurement_duration

    # # projection results
    # proj_res = [(proj * measurement_evo[-1]).tr() for proj in proj]
    # proj_states = [(proj * measurement_evo[-1] * proj.dag()) for proj in proj]
   
    # hada_layer_mod = tensor([hada] * 3)

    # # Correction based on the results 
    # # correction gates
    # cnot1 =  cnot(target=2, control=0, high=2, low=0, N=3)
    # cnot2 =  cnot(target=1, control=0, high=2, low=0, N=3)
    # cnot3 =  cnot(target=0, control=1, high=2, low=0, N=3) 
    # cnot4 =  cnot(target=2, control=1, high=2, low=0, N=3)
    # cnot5 =  cnot(target=0, control=2, high=2, low=0, N=3)
    # cnot6 =  cnot(target=1, control=2, high=2, low=0, N=3)
    
    # # correction_operators
    # r000 = r111 = [[qt.tensor([qt.qeye(dim)] * 3)]]
    # r001 = [[hada_layer_mod], [tensor(tensor([qeye(dim)] * 2), x_gate)], [cnot1], [qt.tensor([qt.qeye(dim)] * 3)], [hada_layer_mod]]
    # r010 = [[hada_layer_mod], [tensor(qeye(dim), x_gate, qeye(dim))], [cnot2], [qt.tensor([qt.qeye(dim)] * 3)], [hada_layer_mod]]
    # r011 = [[hada_layer_mod], [tensor(qeye(dim), x_gate, qeye(dim))], [tensor(tensor([qeye(dim)] * 2), x_gate)], [cnot2], [cnot1], [hada_layer_mod]]
    # r100 = [[hada_layer_mod], [tensor(x_gate, tensor([qeye(dim)] * 2))], [cnot3], [qt.tensor([qt.qeye(dim)] * 3)], [hada_layer_mod]]
    # r101 = [[hada_layer_mod], [tensor(x_gate, tensor([qeye(dim)] * 2))], [tensor(tensor([qeye(dim)] * 2), x_gate)], [cnot3], [cnot4], [hada_layer_mod]]
    # r110 = [[hada_layer_mod], [tensor(x_gate, tensor([qeye(dim)] * 2))], [tensor(qeye(dim), x_gate, qeye(dim))], [cnot5], [cnot6], [hada_layer_mod]]

    # recovery_ops = [r000, r001, r010, r011, r100, r101, r110, r111]
    # recovery_times = [[1.12], [.03 , .03, .5, .53, .03], [.03, .03, .5, .53, .03], [.03, .03, .03, .5, .5, .03], 
    #                   [.03, .03, .5, .53, .03], [.03, .03, .03, .5, .5, .03], [.03, .03, .03, .5, .5, .03], [1.12]]

    # # start the correction procedure 
    # # correction_cycle 
    # corrected_states = []
    # for i in range(len(recovery_ops)):
    #     state_current = qt.ptrace(proj_states[i], [0,1,2])
    #     if proj_res[i] != 0:
    #         for j in range(len(recovery_ops[i])):
    #             trotter_dt = recovery_times[i][j] / 20
    #             trotter = Trotterization(trotter_dt=trotter_dt, T1=T[0], T2=T[1], dim=dim, num_qubits=3, qudit="qutrit")
    #             corrected_state = trotter.apply(state_current, duration=recovery_times[i][j], 
    #                                             unitary=recovery_ops[i][j], errors=True)
    #             state_current = corrected_state[-1]
    #         corrected_states.append(corrected_state)
    #     else:
    #         for j in range(len(recovery_ops[i])):
    #             if type(recovery_ops[i][j]) is not Qobj:
    #                 for k in range(len(recovery_ops[i][j])):
    #                     state_current = recovery_ops[i][j][k] * state_current * (recovery_ops[i][j][k]).dag()
    #             else:
    #                 state_current = recovery_ops[i][j] * state_current * (recovery_ops[i][j]).dag()
    #         corrected_states.append([state_current])
    # total_time += recovery_times[0][0]

    # # combining the corrected states
    # erasure_corrected_state = (sum([proj_res[j] * corrected_states[j][-1] for j in range(len(proj_res))]) / 
    #                            sum([proj_res[j] * corrected_states[j][-1] for j in range(len(proj_res))]).tr())

    
    # return erasure_corrected_state, total_time

In [ ]:
def erasure(values, rho, total_time, state_choice=None):
    
    _, T, dim, N, _, _, _, _, hada, x_gate = values  # experimental values

    hada = qt.Qobj([[1/np.sqrt(2), 0, 1/np.sqrt(2)], [0, 1, 0], [1/np.sqrt(2), 0, -1/np.sqrt(2)]])

    # detection gate creation
    cnot1 = cnot(target=3, control=0, high=1, low=0, N=4)
    cnot2 = cnot(target=4, control=1, high=1, low=0, N=5)
    cnot3 = cnot(target=5, control=2, high=1, low=0, N=6)
    # cnots = [cnot(target=t, control=c, high=1, low=0, N=6) for c,t in [[0,3], [1,4], [2,5]]]

    gates = [cnot1, cnot2, cnot3]
    cnot_time = .5
    gate_times = [cnot_time, cnot_time, cnot_time]

    rho = qt.ptrace(rho, [0,1,2])

    # running the circuit
    for i in range(len(gates)):
        trotter_dt = gate_times[i] / 20
        trotter = Trotterization(trotter_dt=trotter_dt, T1=T[0], T2=T[1], dim=dim, num_qubits= 4 + i, qudit="qutrit")
        rho = tensor(rho, basis(dim, 0) * basis(dim, 0).dag())
        rho_enc = trotter.apply(rho=rho, duration=gate_times[i], unitary=[gates[i]], errors=True)
        rho = rho_enc[-1]
        total_time += gate_times[i]
    
    # measurement
    # measurement operators 
    proj = [tensor(qeye(dim), qeye(dim), qeye(dim), tensor(basis(dim, i), basis(dim, j), basis(dim, k))  
                   * tensor(basis(dim, i), basis(dim, j), basis(dim, k)).dag()) 
             for i in [0,1] for j in [0,1] for k in [0,1]]
    
    measurement_duration = 2
    trotter_dt = measurement_duration / 20
    trotter = Trotterization(trotter_dt=trotter_dt, T1=T[0], T2=T[1], dim=dim, num_qubits=(N + 1), qudit="qutrit")

    measurement_evo = trotter.apply(rho=rho, duration=measurement_duration, unitary=[tensor([qeye(dim)] * (N + 1))], errors=True)
    total_time += measurement_duration

    # projection results
    proj_res = [(proj * measurement_evo[-1]).tr() for proj in proj]
    proj_states = [(proj * measurement_evo[-1] * proj.dag()) for proj in proj]
   
    hada_layer_mod = tensor([hada] * 3)

    # Correction based on the results 
    # correction gates
    cnot1 =  cnot(target=2, control=0, high=2, low=0, N=3)
    cnot2 =  cnot(target=1, control=0, high=2, low=0, N=3)
    cnot3 =  cnot(target=0, control=1, high=2, low=0, N=3) 
    cnot4 =  cnot(target=2, control=1, high=2, low=0, N=3)
    cnot5 =  cnot(target=0, control=2, high=2, low=0, N=3)
    cnot6 =  cnot(target=1, control=2, high=2, low=0, N=3)
    
    # correction_operators
    r000 = r111 = [[qt.tensor([qt.qeye(dim)] * 3)]]
    r001 = [[hada_layer_mod], [tensor(tensor([qeye(dim)] * 2), x_gate)], [cnot1], [hada_layer_mod]]
    r010 = [[hada_layer_mod], [tensor(qeye(dim), x_gate, qeye(dim))], [cnot2], [hada_layer_mod]]
    r011 = [[hada_layer_mod], [tensor(qeye(dim), x_gate, qeye(dim))], [tensor(tensor([qeye(dim)] * 2), x_gate)], [cnot2], [cnot1], [hada_layer_mod]]
    r100 = [[hada_layer_mod], [tensor(x_gate, tensor([qeye(dim)] * 2))], [cnot3], [hada_layer_mod]]
    r101 = [[hada_layer_mod], [tensor(x_gate, tensor([qeye(dim)] * 2))], [tensor(tensor([qeye(dim)] * 2), x_gate)], [cnot3], [cnot4], [hada_layer_mod]]
    r110 = [[hada_layer_mod], [tensor(x_gate, tensor([qeye(dim)] * 2))], [tensor(qeye(dim), x_gate, qeye(dim))], [cnot5], [cnot6], [hada_layer_mod]]

    recovery_ops = [r000, r001, r010, r011, r100, r101, r110, r111]
    recovery_times = [[0.0], [.03 , .03, .5, .03], [.03, .03, .5, .03], [.03, .03, .03, .5, .5, .03], 
                      [.03, .03, .5, .03], [.03, .03, .03, .5, .5, .03], [.03, .03, .03, .5, .5, .03], [0.0]]

    # start the correction procedure 
    # correction_cycle 
    corrected_states = []
    for i in range(len(recovery_ops)):
        state_current = qt.ptrace(proj_states[i], [0,1,2])
        if proj_res[i] != 0 and i not in [0, len(recovery_ops) - 1]:
            for j in range(len(recovery_ops[i])):
                trotter_dt = recovery_times[i][j] / 20
                trotter = Trotterization(trotter_dt=trotter_dt, T1=T[0], T2=T[1], dim=dim, num_qubits=3, qudit="qutrit")
                corrected_state = trotter.apply(state_current, duration=recovery_times[i][j], 
                                                unitary=recovery_ops[i][j], errors=True)
                state_current = corrected_state[-1]
            corrected_states.append(corrected_state)
        else:
            for j in range(len(recovery_ops[i])):
                if type(recovery_ops[i][j]) is not Qobj:
                    for k in range(len(recovery_ops[i][j])):
                        state_current = recovery_ops[i][j][k] * state_current * (recovery_ops[i][j][k]).dag()
                else:
                    state_current = recovery_ops[i][j] * state_current * (recovery_ops[i][j]).dag()
            corrected_states.append([state_current])

    for i in range(len(proj_res)):
        total_time += proj_res[i] * sum(recovery_times[i])

    # combining the corrected states
    erasure_corrected_state = (sum([proj_res[j] * corrected_states[j][-1] for j in range(len(proj_res))]) / 
                               sum([proj_res[j] * corrected_states[j][-1] for j in range(len(proj_res))]).tr())

    
    return erasure_corrected_state, total_time

In [ ]:
from quantum_logical.state import state as st

In [ ]:
hada = qt.Qobj([[1/np.sqrt(2), 0, 1/np.sqrt(2)], [0, 1, 0], [1/np.sqrt(2), 0, -1/np.sqrt(2)]])
# vector setup 
basis0 = qt.Qobj([[1],[0],[0]])
basis1 = qt.Qobj([[0],[1],[0]])
basis2 = qt.Qobj([[0],[0],[1]])
vector0 = hada * basis0
vector1 = hada * basis1
vector2 = hada * basis2
vectors = [vector0, vector1, vector2]
vectors = [tensor(i,j,k) for i in vectors for j in vectors for k in vectors]

In [ ]:
N = 5
dim = 3
cnots, correction_z, hada_layer, x_layer, correction_x, x_gate, hada, x_gate_ = gate(dim=dim, N=N)

In [ ]:
# iterations = 1
# # building T1 and T2 lists 
# t1_list = np.linspace(100, 160, iterations)
# t_list = []
# for i in range(len(t1_list)):
#     t2s = np.linspace(t1_list[i] * (2/3), t1_list[i] * (2/3), 1)
#     for j in range(len(t2s)):
#         t_list.append([t1_list[i], t2s[j]])

# values = []
# for i in range(iterations):
#     values.append([cnots, t_list[i], dim, N, x_gate, x_layer, correction_z, hada_layer, hada, x_gate_])

In [ ]:
# import random

In [ ]:
# alpha = []
# beta = []
# state_choices = []
# num_states = 100
# for i in range(num_states):
#     alpha = complex(random.uniform(-10, 10), random.uniform(-10, 10))
#     beta = complex(random.uniform(-10, 10), random.uniform(-10, 10))
#     state_choices.append([["+","+","+"], alpha, beta])




In [ ]:
iterations = 50
t1_list = np.linspace(.1, 200, iterations)
# t1_list = np.linspace(100, 100, iterations)
t_list = []
for i in range(len(t1_list)):

    t2s = np.linspace(t1_list[i] * (2/3), 2 * t1_list[i] - 1, 1)
    
    for j in range(len(t2s)):
        t_list.append([t1_list[i], t2s[j]])

values = []
for i in range(iterations):
    values.append([cnots, t_list[i], dim, N, x_gate, x_layer, correction_z, hada_layer, hada, x_gate_])

physical_error = []
logical_error = []
phase_errors = []
erasure_errors = []
t1 = []
t2 = []
physical_err = []
times = []

cycles = 1

state_choices = [[["-", "-", "-"], 1, 0], [["+", "+", "+"], 1, 0], 
                     [["+", "+", "+"], 1, 1], [["+", "+", "+"], 1, -1], 
                     [["+", "+", "+"], 1, 1j], [["+", "+", "+"], 1, -1j]]

proj_res = []

for state_choice in state_choices:
    log_err = []
    phys_err = []
    time = []
    proj_res_ = []
    # eras_err = []
    # phase_err = []
    physical_rate = []


    # this has to be the ugliest code you have ever written (fix this)
    # if state_choice == [["-", "-", "-"], 1, 0]:
    #     vectors_phase = []
        # vectors_erasure = []
        # rho, state = st(qubit_choices=["-", "-", "+"], dim=3, alpha=state_choice[1], beta=state_choice[2])
        # vectors_phase.append(state)
        # rho, state = st(qubit_choices=["-", "+", "-"], dim=3, alpha=state_choice[1], beta=state_choice[2])
        # vectors_phase.append(state)
        # rho, state = st(qubit_choices=["+", "-", "-"], dim=3, alpha=state_choice[1], beta=state_choice[2])
        # vectors_phase.append(state)
        # rho, state = st(qubit_choices=["-", "-", "1"], dim=3, alpha=state_choice[1], beta=state_choice[2])
        # vectors_erasure.append(state)
        # rho, state = st(qubit_choices=["-", "1", "1"], dim=3, alpha=state_choice[1], beta=state_choice[2])
        # vectors_erasure.append(state)
        # rho, state = st(qubit_choices=["1", "-", "-"], dim=3, alpha=state_choice[1], beta=state_choice[2])
        # vectors_erasure.append(state)
        # rho, state = st(qubit_choices=["1", "-", "1"], dim=3, alpha=state_choice[1], beta=state_choice[2])
        # vectors_erasure.append(state)
        # rho, state = st(qubit_choices=["1", "1", "-"], dim=3, alpha=state_choice[1], beta=state_choice[2])
        # vectors_erasure.append(state)
        # rho, state = st(qubit_choices=["-", "1", "-"], dim=3, alpha=state_choice[1], beta=state_choice[2])
        # vectors_erasure.append(state)
    # elif state_choice[0] == ["+", "+", "+"]:
    #     vectors_phase = []
    #     rho, state = st(qubit_choices=["-", "+", "+"], dim=3, alpha=state_choice[1], beta=state_choice[2])
    #     vectors_phase.append(state)
    #     rho, state = st(qubit_choices=["+", "+", "-"], dim=3, alpha=state_choice[1], beta=state_choice[2])
    #     vectors_phase.append(state)
    #     rho, state = st(qubit_choices=["+", "-", "+"], dim=3, alpha=state_choice[1], beta=state_choice[2])
    #     vectors_phase.append(state)
    #     rho, state = st(qubit_choices=["+", "+", "1"], dim=3, alpha=state_choice[1], beta=state_choice[2])
        # vectors_erasure.append(state)
        # rho, state = st(qubit_choices=["+", "1", "1"], dim=3, alpha=state_choice[1], beta=state_choice[2])
        # vectors_erasure.append(state)
        # rho, state = st(qubit_choices=["1", "+", "+"], dim=3, alpha=state_choice[1], beta=state_choice[2])
        # vectors_erasure.append(state)
        # rho, state = st(qubit_choices=["1", "+", "1"], dim=3, alpha=state_choice[1], beta=state_choice[2])
        # vectors_erasure.append(state)
        # rho, state = st(qubit_choices=["1", "1", "+"], dim=3, alpha=state_choice[1], beta=state_choice[2])
        # vectors_erasure.append(state)
        # rho, state = st(qubit_choices=["+", "1", "+"], dim=3, alpha=state_choice[1], beta=state_choice[2])
        # vectors_erasure.append(state)

    for value in tqdm(values):
    
        rho_encoded, state_vector_ = st(qubit_choices=state_choice[0], dim=3, alpha=state_choice[1], beta=state_choice[2])

        order = [0, 0, 1]

        total_time = 0
        for i in range(cycles):
            for choice in order:
                if choice == 0:
                    rho_encoded, total_time = repetition(values=value, rho=rho_encoded, total_time=total_time)

                elif choice == 1:
                    rho_encoded, total_time = erasure(rho=rho_encoded, values=value, total_time=total_time)

        def neilson_fid(rho, sigma):
            return (((rho.sqrtm()) * sigma * (rho.sqrtm())).sqrtm()).tr()

        # logical = [state_vector_]
        # vals = []
        # for vec in logical:
        #     val = (vec.dag() * qt.ptrace(rho_encoded, [0,1,2]) * vec)[0][0][0]
        #     vals.append(np.abs(val))
        # log_err.append(np.abs(1 - np.abs(sum(vals))))
        log_err.append(np.abs(neilson_fid(rho=qt.ptrace(rho_encoded, [0,1,2]), sigma=state_vector_ * state_vector_.dag())))

        # vals = []
        # for vec in vectors_phase:
        #     val = (vec.dag() * qt.ptrace(rho_encoded, [0,1,2]) * vec)[0][0][0]
        #     vals.append(np.abs(val))
        # phase_err.append(np.abs(sum(vals)))

        # vals = []
        # for vec in vectors_erasure:
        #     val = (vec.dag() * qt.ptrace(rho_encoded, [0,1,2]) * vec)[0][0][0]
        #     vals.append(np.abs(val))
        # eras_err.append(np.abs(sum(vals)))

        # vals = []
        # for vec in vectors:
        #     val = (vec.dag() * qt.ptrace(rho_encoded, [0,1,2]) * vec)[0][0][0]
        #     vals.append(np.abs(val))

        t_phase = (2 * value[1][0] * value[1][1])/(2 * value[1][0] - value[1][1])
        physical_error_val = ((1 - np.exp((-total_time) * ((1/value[1][0])))) + (1 - np.exp((-total_time) * ((1/t_phase)))) 
                            - (1 - np.exp((-total_time) * ((1/value[1][0])))) * (1 - np.exp((-total_time) * ((1/t_phase)))))
        phys_err.append(np.abs(physical_error_val))
        time.append(total_time)
        phys_rate = 1/(value[1][0]) + 1/((2*value[1][0]*value[1][1])/(2*value[1][0] - value[1][1]))
        physical_rate.append(phys_rate)

    # phase_errors.append(phase_err)
    # erasure_errors.append(eras_err)
    physical_error.append(phys_err)
    logical_error.append(log_err)
    times.append(time)

100%|██████████| 50/50 [06:48<00:00,  8.16s/it]


In [ ]:
# sort data
phys = []
logic = []
time_list = []
# e_errors = []
# p_errors = []

for i in range(len(values)):
    phys.append(sum([physical_error[j][i] for j in range(len(state_choices))]) / len(state_choices))
    logic.append(sum([logical_error[j][i] for j in range(len(state_choices))]) / len(state_choices))
    time_list.append(np.abs(sum([times[j][i] for j in range(len(state_choices))]) / len(state_choices)))
    # e_errors.append(sum([erasure_errors[j][i] for j in range(len(state_choices))]) / len(state_choices))
    # p_errors.append(sum([phase_errors[j][i] for j in range(len(state_choices))]) / len(state_choices))
# for i in range(cycles):
#     phys.append(sum([physical_error[j][i] for j in range(len(state_choices))]) / len(state_choices))
#     logic.append(sum([logical_error[j][i] for j in range(len(state_choices))]) / len(state_choices))
#     time_list.append(np.abs(sum([times[j][i] for j in range(len(state_choices))]) / len(state_choices)))
    # e_errors.append(sum([erasure_errors[j][i] for j in range(len(state_choices))]) / len(state_choices))
    # p_errors.append(sum([phase_errors[j][i] for j in range(len(state_choices))]) / len(state_choices))

In [ ]:
import csv

# Writing the arrays to a CSV file
with open('3rd_order_001_t1_fixed_diff.csv', 'w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(phys)  
    writer.writerow(logic)
    writer.writerow(time_list)
    # writer.writerow(e_errors)  
    # writer.writerow(p_errors)   
    writer.writerow(physical_rate)   